# Phase 8: Train and Evaluate the PinchPy ANN Model

In this notebook, we:
1. Load the physically verified synthetic dataset of 1,000 cases.
2. Train an Artificial Neural Network (Multilayer Perceptron) to surrogate the exact Pinch Analysis targeting engine.
3. Evaluate the model performance using $R^2$, MAE, and RMSE metrics across all three targets ($Q_{H,\min}$, $Q_{C,\min}$, $T_{\text{pinch}}$).
4. Test direct stream-level inference with `predict_case` and compare with exact calculation.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from pinchpy.models.stream import Stream
from pinchpy.ml.train import train_ann
from pinchpy.ml.predict import predict_targets, predict_case
from pinchpy.ml.evaluate import evaluate_regression
from pinchpy.ml.features import TARGET_NAMES
from pinchpy.analysis.pinch_point import utility_targets

## 1. Load Dataset and Split Train/Test

In [ ]:
csv_path = Path("../data/raw/pinch_dataset_1000.csv")
df = pd.read_csv(csv_path)

X = df.iloc[:, :-3]
y = df.iloc[:, -3:]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape:     {X_test.shape}")

## 2. Train the Feed-Forward ANN
The ANN uses a `StandardScaler` preprocessor chained into a two-layer MLP (`128 -> 64`) with ReLU activation, Adam optimizer, and early stopping.

In [ ]:
model = train_ann(
    X_train,
    y_train,
    hidden_layer_sizes=(128, 64),
    max_iter=1000,
    random_state=42,
)
print("Model trained successfully!")

## 3. Evaluate Predictions on Unseen Test Set

In [ ]:
y_pred = predict_targets(model, X_test)
metrics = evaluate_regression(y_test, y_pred)

print("=== Overall Model Performance ===")
print(f"  R² Score : {metrics['r2']:.4f}")
print(f"  MAE      : {metrics['mae']:.2f}")
print(f"  RMSE     : {metrics['rmse']:.2f}")

print("\n=== Per-Target Breakdown ===")
for target, m in metrics["per_target"].items():
    print(f"[{target}]")
    print(f"  R²  = {m['r2']:.4f}")
    print(f"  MAE = {m['mae']:.2f}")
    print(f"  RMSE= {m['rmse']:.2f}")

## 4. Parity Plots (Predicted vs Exact Ground Truth)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
y_test_np = y_test.values

for i, (ax, name) in enumerate(zip(axes, TARGET_NAMES)):
    actual = y_test_np[:, i]
    predicted = y_pred[:, i]
    r2 = metrics["per_target"][name]["r2"]
    
    ax.scatter(actual, predicted, alpha=0.5, color="steelblue", edgecolor="none", s=30)
    lims = [min(actual.min(), predicted.min()), max(actual.max(), predicted.max())]
    ax.plot(lims, lims, "r--", lw=1.5, label="Ideal (1:1)")
    ax.set_title(f"{name}\n$R^2 = {r2:.3f}$")
    ax.set_xlabel("Exact Value")
    ax.set_ylabel("ANN Prediction")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Direct Stream-Level Inference vs Exact Engine
Let's test an arbitrary 4-stream network with `predict_case` and compare it side-by-side with the exact thermodynamic solver.

In [ ]:
sample_streams = [
    Stream("H1", "hot", 170, 60, 3.0),
    Stream("H2", "hot", 150, 30, 1.5),
    Stream("C1", "cold", 20, 135, 2.0),
    Stream("C2", "cold", 80, 140, 4.0),
]

dtmin = 10.0
exact = utility_targets(sample_streams, dtmin)
predicted = predict_case(model, sample_streams, dtmin)

print(f"{'Metric':<25} {'Exact Engine':>15} {'ANN Surrogate':>15} {'Error':>12}")
print("-" * 70)
for target in ["minimum_hot_utility", "minimum_cold_utility", "pinch_temperature"]:
    ex = exact[target] if target in exact else exact["pinch_shifted_temperature"]
    pr = predicted[target]
    print(f"{target:<25} {ex:15.2f} {pr:15.2f} {abs(pr - ex):12.2f}")